In [1]:
import pandas as pd
import numpy as np

from sklearn import preprocessing
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from xailib.data_loaders.dataframe_loader import prepare_dataframe

from xailib.explainers.lime_explainer import LimeXAITabularExplainer
from xailib.explainers.lore_explainer import LoreTabularExplainer
from xailib.explainers.shap_explainer_tab import ShapXAITabularExplainer

from xailib.models.sklearn_classifier_wrapper import sklearn_classifier_wrapper

import altair as alt
import pickle

import os

ModuleNotFoundError: No module named 'lore'

- Riscrivere il codice con due funzioni:
    - una per il preprocessing dei dati
    - una per il plotting
- Check su cosa prendere per le regole e le soglie
- Usare la FI per ordinare le feature  [X]
    - hconcat con la FI (plot accanto a plot)  [X]
- aggiungere il cutoff su entrambi
- inserire progressive disclosure:
    - filtrare per feature a cui è associata Rules
    - filtrare per FI (cutoff)

- Investigare le CR
- Fare la stessa cosa con titanic [X]

- nel paper partire dalla vecchia viz html (linguaggio naturale)

In [2]:
path = os.getcwd()
print(path)

In [3]:
source_file = '../datasets/titanic_c.csv'
class_field = 'Survived'
df = pd.read_csv(source_file, skipinitialspace=True, na_values='?', keep_default_na=True)

In [4]:
source_file = '../datasets/german_credit.csv'
class_field = 'default'
# Load and transform dataset
df = pd.read_csv(source_file, skipinitialspace=True, na_values='?', keep_default_na=True)

In [5]:
df, feature_names, class_values, numeric_columns, rdf, real_feature_names, features_map = prepare_dataframe(df, class_field)

In [6]:
inv_dict=[]
for i,el in enumerate(features_map.values()):
    #invert key value in el dict
    d = {v: k for k, v in el.items()}
    for k, v in d.items():
        inv_dict.append(d[k])

### Learning a Random Forest classfier

We train a RF classifier by using the ```sklearn``` library. We start by splitting the dataset into a train and test subsets. 

In [7]:
test_size = 0.3
random_state = 42
X_train, X_test, Y_train, Y_test = train_test_split(df[feature_names], df[class_field],
                                                        test_size=test_size,
                                                        random_state=random_state,
                                                        stratify=df[class_field])


Then we train the model on the training set. 
Once the model has been learned, we use a wrapper class to get access to the model for ```XAI lib```

In [8]:
bb = RandomForestClassifier(n_estimators=20, random_state=random_state)
bb.fit(X_train.values, Y_train.values)
bbox = sklearn_classifier_wrapper(bb)

Select a new instance to be classfied by the model and print the predicted class.

In [9]:
inst = X_train.iloc[128].values
print('Instance ',inst)
print('True class ',Y_train.iloc[128])
print('Predicted class ',bb.predict(inst.reshape(1, -1)))

In [10]:
real_inst = inst
real_inst

## Explaining the prediction
We use the explanators of ```XAI lib``` to provide an explantion for the classified instance ```inst```.
Every explainer of ```XAI lib``` takes in input the blackbox to be explained with the corresponding feature names, and a configuration object to initialize the explainer.

### SHAP explainer

In [11]:
explainer = ShapXAITabularExplainer(bbox, feature_names)
config = {'explainer' : 'tree', 'X_train' : X_train.iloc[0:].values}
explainer.fit(config)

In [12]:
exp = explainer.explain(inst)

In [13]:
exp.exp

In [14]:
shap_feature_importance=exp.exp
shap_feature_importance

In [15]:
exp.plot_features_importance()

## Learning a different model

### Learning a Logistic Regressor

We train a Logistic Regression by using the ```sklearn``` library. We transform the dataset by using a ```Scaler``` to normalize all the attributes.

In [16]:
scaler = preprocessing.StandardScaler().fit(X_train)
X_scaled = scaler.transform(X_train)

bb = LogisticRegression(C=1, penalty='l2')
bb.fit(X_scaled, Y_train.values)
# pass the model to the wrapper to use it in the XAI lib
bbox = sklearn_classifier_wrapper(bb)

In [17]:
# select a record to explain
inst = X_scaled[182]
print('Instance ',inst)
print('Predicted class ',bb.predict(inst.reshape(1, -1)))

In [18]:
X_scaled

## Explaining the prediction
We use the same explainators as for the previous model. In this case, a few adjustments are necessary for the initialization of the explanators. For example, SHAP needs a specific configuration for the linear model we are using.

## LIME tabular explainer

In [19]:
limeExplainer = LimeXAITabularExplainer(bbox)
config = {'feature_selection': 'lasso_path'}
limeExplainer.fit(df, class_field, config)
lime_exp = limeExplainer.explain(inst)
print(lime_exp.exp.as_list())# è una lista di tuple

In [20]:
lime_feature_imp=lime_exp.exp.as_list()
lime_feature_imp

In [21]:
lime_exp.plot_features_importance()

### LORE explainer

In [22]:
explainer = LoreTabularExplainer(bbox)
config = {'neigh_type':'geneticp', 'size':1000, 'ocr':0.1, 'ngen':10}
explainer.fit(df, class_field, config)
exp = explainer.explain(inst)
print(exp)

In [23]:
exp.plotRules()

In [24]:
exp.plotCounterfactualRules()

In [25]:
rules =exp.expDict['rule']['premise']

In [26]:
rules

In [27]:
for r in rules:
    print(r['att'])

In [28]:
df_rules = pd.DataFrame.from_records(rules)

In [29]:
df.describe()

In [30]:
df_range=pd.concat(
    {'min':X_train.min(),
     'max':X_train.max(),
     'std':X_train.std(),
     'q1':X_train.quantile(0.25),
     'median':X_train.quantile(0.50),
     'q3':X_train.quantile(0.75),
     },axis=1)

In [31]:
df_range=df_range.reset_index()

In [32]:
df_range

In [33]:
df_viz = df_range.merge(df_rules,how='left',left_on='index',right_on='att')
df_viz = df_viz.drop('att', axis=1)
df_viz

In [34]:
thr2_list=[]
for i, row in df_viz.iterrows():
    if row['op']== '>' or row['op']== '>=':
        thr2_list.append(row['max'])
        continue
    if row['op']== '<' or row['op']== '<=':
        thr2_list.append(row['min'])
        continue
    else:
        thr2_list.append(np.nan)
df_viz['thr2'] = thr2_list
df_viz

In [35]:
for string, value in lime_feature_imp:
    print(value)
    break

In [36]:
def add_fi_to_df_and_sort(dataframe, values):
    for string, value in values:
        dataframe.loc[dataframe['index'] == string, 'feature_importance'] = value
        dataframe.sort_values(by=['feature_importance'], key=lambda x: abs(x), ascending=False, inplace=True)
    return dataframe
df_fi=add_fi_to_df_and_sort(df_viz,lime_feature_imp)
df_fi

In [37]:
df_viz['inst'] = real_inst.tolist()
df_viz

In [38]:
features=df_viz['index'].to_list()
features

In [39]:
df_rules = pd.DataFrame.from_records(rules)
df_rules

In [40]:
def single_rule_plot_n(dataframe, rw):
    p=alt.Chart(
        dataframe[dataframe['index'] == rw['index']]
    ).mark_point(
        color='black' if rw['is_continuous'] == True else 'black',
        size=20,
        shape='diamond'
    ).encode(
        x=alt.X(
            field='inst',
            type='quantitative',
            title=None,
            scale= alt.Scale(domain=(rw['min'], rw['max']), clamp=True, nice=False)
        ),
        tooltip=[alt.Tooltip(field='inst', title=rw['index'])]
    )

    t_min = alt.Chart(
        dataframe[dataframe['index'] == rw['index']]
    ).mark_text(
        color='black',
        dx=-10,
        align='right'
    ).encode(
        x=alt.X(
            field='min',
            type='quantitative',
            title=None
        ),
        text='min:N'
    )

    t_max = alt.Chart(
        dataframe[dataframe['index'] == rw['index']]
    ).mark_text(
        color='black',
        dx=10,
        align='left'
    ).encode(
        x=alt.X(
            field='max',
            type='quantitative',
            title=None
        ),
        text='max:N'
    )

    q1_m = alt.Chart(
        dataframe[dataframe['index'] == rw['index']]
    ).mark_bar(
        color='#DAE7E8',
        size=12
    ).encode(
        x=alt.X(
            field='q1',
            type='quantitative',
            title=None,
            scale= alt.Scale(domain=(rw['min'], rw['max']), clamp=True, nice=False)
        ),
        x2 = alt.X2(
            field='median'
        ),
    )

    m_q3 = alt.Chart(
        dataframe[dataframe['index'] == rw['index']]
    ).mark_bar(
        color='#A8B9BF',
        size=12
    ).encode(
        x=alt.X(
            field='median',
            type='quantitative',
            title=None,
            scale= alt.Scale(domain=(rw['min'], rw['max']), clamp=True, nice=False)
        ),
        x2 = alt.X2(
            field='q3'
        ),
    )

    b =alt.Chart(
        dataframe[dataframe['index'] == rw['index']]
    ).mark_bar(
        color='#f28e46',size=5
    ).encode(
        x=alt.X(
            field='thr',
            type='quantitative',
            title=None,
        ),
        x2='thr2',
        y=alt.Y(
            field='index',
            type='nominal',
            title=None
        ),

    )



    l =alt.Chart(
        dataframe[dataframe['index'] == rw['index']]
    ).mark_bar(
        color='grey',size=1
    ).encode(
        x=alt.X(
            field='min',
            type='quantitative',
            title=None,
            scale= alt.Scale(domain=(rw['min'], rw['max']), clamp=True, nice=False)
        ),
        x2='max',
        y=alt.Y(field='index',type='nominal',title=None, axis=alt.Axis(labels= False, ticks=False))
    )


    return alt.layer(l,q1_m,m_q3,b,t_min,t_max,p).properties(
        height=12,
        width=399,
    )

In [41]:
def single_feature_importance_plot(dataframe, rw):

    chart = alt.Chart(
        dataframe[dataframe['index'] == rw['index']]
    ).mark_bar(
    ).encode(
        x=alt.X(
            field='feature_importance',
            type='quantitative',
            title=None,
            scale=alt.Scale(
                domain=(dataframe['feature_importance'].min(), dataframe['feature_importance'].max()),
                nice=False
            )
        ),
        y=alt.Y(
            field='index',
            type='nominal',
            title=None,
            axis=None
        ),
        color=alt.condition('datum.feature_importance > 0', alt.value('#2C0AD1'), alt.value('#DB2C8F') ),
    )
    return chart.properties(
        height=12,
        width=50
    )

In [42]:
def single_instance_text(dataframe, rw):
    chart = alt.Chart(
        dataframe[dataframe['index'] == rw['index']]
    ).mark_text(
        color='black',
        align='left',
        dx=-30,
        fontWeight='bold'
    ).encode(
            text=alt.Text(
            field='inst',
            type='quantitative',
            title=None
        )
    )
    return chart.properties(
        height=12,
        width=10
    )

In [43]:
def single_index_text(dataframe, rw):
    chart = alt.Chart(
        dataframe[dataframe['index'] == rw['index']]
    ).transform_calculate(
        label ="datum.type=='categorical' ? datum.index : datum.index +' = '+ datum.inst" #  datum.index +' = '+ datum.inst
    ).mark_text(
        color='black',
        align='left',
        dx=-100,
    ).encode(
            text=alt.Text(
            field='label',
            type='nominal',
            title=None
        )
    )
    return chart.properties(
        height=12,
        width=50
    )

In [44]:
def single_rule_plot_q(dataframe, rw):
    name= rw['index'].split('=')[0]
    data = dataframe[dataframe['rname'] == name]
    b = alt.Chart(
        data
    ).mark_bar(
        stroke='white'
    ).encode(
        x=alt.X(
            field='count',
            type='quantitative',
            title=None,
            # stack="normalize"
        ),
        y=alt.Y(
            field='rname',
            type='nominal',
            axis=None
        ),
        detail='index:N',
        color=alt.condition('datum.inst==1',alt.value('darkgrey'),alt.value('lightgrey'))
    )

    r=alt.Chart(
        data
    ).mark_bar(
        stroke='white'
    ).encode(
        x=alt.X(
            field='count',
            type='quantitative',
            title=None,
            # stack="normalize"
        ),
        y=alt.Y(
            field='rname',
            type='nominal',
            axis=None
        ),
        detail='index:N',
        color=alt.condition('datum.is_continuous',alt.value('#f28e46'),alt.value('white')),
        opacity=alt.condition('datum.is_continuous',alt.value(1),alt.value(0.001)),
        tooltip=[alt.Tooltip(field='category', title=name), alt.Tooltip(field='count')]
    )
    
    dot =alt.Chart(
        data
    ).transform_stack(
        stack='count',
        as_=['count_start','count_end'],
        groupby=['rname'],
        sort=[alt.SortField('count', 'descending')]
    ).mark_point(
        size=40,
        shape='diamond',
        color='black'
    ).encode(
        x=alt.X(
            field='count_start',
            type='quantitative',
            title=None,
            # stack="normalize",
        ),
        x2=alt.X2(
            field='count_end',
            type='quantitative',
            title=None,
            # stack="normalize",
        ),
        y=alt.Y(
            field='rname',
            type='nominal',
            axis=None
        ),
        detail='index:N',
        # color=alt.condition('datum.is_continuous',alt.value('#f28e46'),alt.value('white')),
        # opacity=alt.condition('datum.is_continuous',alt.value(1),alt.value(0.001)),
        #tooltip=[alt.Tooltip(field='category', title=name), alt.Tooltip(field='count')]
    )
    

    return alt.layer(b,r).properties(
        height=10,
        width=400,
    )

In [45]:
def plot_rules(dataframe):
    tx_list=[]
    ti_list=[]
    rp_list=[]
    fi_list=[]
    for i, row in dataframe.iterrows():
        if row['inst']!=0: # or row['is_continuous']==True
            stx = single_instance_text(dataframe, row)
            sti = single_index_text(dataframe, row)
            if row['type']== 'numeric':
                srp = single_rule_plot_n(dataframe, row)
            else:
                srp = single_rule_plot_q(dataframe, row)
            sfi = single_feature_importance_plot(dataframe, row)
            tx_list.append(stx)
            ti_list.append(sti)
            rp_list.append(srp)
            fi_list.append(sfi)
    tx_concat=alt.vconcat(*tx_list)
    ti_concat=alt.vconcat(*ti_list)
    rp_concat=alt.vconcat(*rp_list, title='Rule')
    fi_concat=alt.vconcat(*fi_list, title='FI')
    final_chart = alt.hconcat(fi_concat, rp_concat, ti_concat)

    return final_chart.configure_concat(
        spacing=3
    ).configure_axis(
        grid=False
    ).configure_view(
        strokeWidth=0,
        stroke='lightgray'
    ).configure_axisX(
        disable=True
    ).configure_axisY(
        domain=False,
        ticks=False
    ).configure_title(
        fontWeight='bold',
        anchor='middle',

    )

In [46]:
def dot_on_categories(dataframe,sort_by_rules=True):
    base =alt.Chart(
        dataframe
    ).transform_stack(
        stack='count',
        as_=['count_start','count_end'],
        groupby=['rname'],
        sort=[alt.SortField('count', 'ascending')]
    ).transform_calculate(
        midStack='(datum.count_start+datum.count_end)/2'
    )
    
    bar = base.mark_bar(stroke='white').encode(
        x='count_start:Q',
        x2='count_end:Q',
        y=alt.Y('rname:N',sort=["is_continuous", "feature_importance"]),
        detail='index:N',
        color=alt.condition('datum.is_continuous',alt.value("#f4dd4d"),alt.value('lightgrey')),
        tooltip=[alt.Tooltip('category')]
    )
    
    dot= base.mark_point(
        color='black',
        shape='diamond',
        fill='black',
        size=30
    ).encode(
        x='midStack:Q',
        y=alt.Y('rname:N',sort=["is_continuous", "feature_importance"]),
        detail='index:N',
        opacity=alt.condition('datum.inst==1',alt.value(0.6),alt.value(0))
    )

    return (bar+dot).properties(
        width=200,
    ).configure_axisX(
        disable=True
    ).configure_axisY(
        labelPadding=10,
        labelFontSize=12,
        domain=False,
        ticks=False,
        title=None
    ).configure_axis(
        grid=True
    ).configure_view(
        strokeWidth=0,
        stroke='lightgray'
    )
#dot_on_categories(df_viz)

# Function to prepare data for plotting

In [47]:
def data_to_plot(
        feature_names=feature_names, real_feature_names=real_feature_names,
        instance_number=None, x_train=None, rules=None,
        feature_importance=None):
    feature_list =[]
    #Convert the list of tuples generated by lime in a dict
    if feature_importance is 'lime':
        lime_dict = {}
        for (key, value) in lime_feature_importance:
            lime_dict.setdefault(key, value)
    for i, el in enumerate(feature_names):
        f ={}
        if el in numeric_columns:
            f['type'] = 'numeric'
            f['name'] = el
            f['rname'] = real_feature_names[i]
            if x_train is not None:
                f['min'] = x_train[el].min()
                f['max'] = x_train[el].max()
                f['q1'] = x_train[el].quantile(0.25)
                f['median'] = x_train[el].quantile(0.50)
                f['q3'] = x_train[el].quantile(0.75)
                f['mean'] = x_train[el].mean()
                f['std'] = x_train[el].std()
        else:
            f['type'] = 'categorical'
            f['name'] = el
            f['rname'] = el.split('=')[0]
            f['category'] = el.split('=',1)[1]
            if x_train is not None:
                f['count'] = x_train[el].sum()

        if feature_importance is 'lime':
            f['feature_importance'] = lime_dict[el]
        if feature_importance is 'shap':
            f['feature_importance'] = shap_feature_importance[1][i]
        feature_list.append(f)
    df =pd.DataFrame.from_records(feature_list)

    if instance_number:
        inst = X_train.iloc[instance_number].values
        df['inst'] = inst
    if rules is not None:
        df_rules = pd.DataFrame.from_records(rules)
        df = df.merge(df_rules,how='left',left_on='name',right_on='att')
        df = df.drop('att', axis=1)
        thr2_list=[]
        for i, row in df.iterrows():
            if row['op']== '>' or row['op']== '>=':
                thr2_list.append(row['max'])
                continue
            if row['op']== '<' or row['op']== '<=':
                thr2_list.append(row['min'])
                continue
            else:
                thr2_list.append(np.nan)
        df['thr2'] = thr2_list
    df.sort_values(by=['feature_importance'], key=lambda x: abs(x), ascending=False, inplace=True)
    return df

In [48]:
df_v = data_to_plot(feature_names=feature_names, real_feature_names=real_feature_names, instance_number=3, x_train=X_train, rules=rules, feature_importance='shap')
df_v

In [49]:
plot_rules(df_v.rename(columns={'name':'index'}))

# DATI NARET

In [50]:
pip install xgboost

In [51]:
pip install dill

In [52]:
pip install category-encoders

In [53]:
import xgboost as xgb
import dill
import SuperLore
import category_encoders

### TRAIN MODEL

In [54]:
bb = xgb.XGBClassifier()
bb.load_model("../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_train.model")


In [55]:
bb

### X_train

In [56]:
X_train = pd.read_pickle('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_xtrain')
X_train

### Y_train

In [57]:
Y_train = pd.read_pickle('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_ytrain')
Y_train

### X_test

In [58]:
X_test = pd.read_pickle('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_xtest')
X_test

### Y_test

In [59]:
Y_test = pd.read_pickle('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_ytest')
Y_test

In [60]:
Y_test

### Data description

In [61]:
data_desc=pd.read_pickle('../datasets/Dati-Banca-Lore/intesa_incassi_data_description.p')
data_desc

### Lore exp

In [62]:
lore_exp_path=open('../datasets/Dati-Banca-Lore/lore_exp_INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_xgb_cfs_binary_from_dts.p','rb')

In [63]:
objs = []
while 1:
    try:
        objs.append(pickle.load(lore_exp_path))
    except EOFError:
        break

In [64]:
objs

In [65]:
print(objs[0])

for c in X_train.columns:
        if X_train[c].max() == 1.0 and X_train[c].min() == 0.0:
            print('Colonna categorica! ', c)
        else:
            numeric_columns.append(c)

In [66]:
prova = objs[0].rule.class_name
prova

class Rule(object):

    def __init__(self, premises, cons, class_name):
        self.premises = premises
        self.cons = cons
        self.class_name = class_name

In [67]:
for p in prova:
    print(p.att)
    print(p.op)

In [ ]:
print(objs[1])

In [ ]:
for i in objs[3].crules:
    print(i)

### SHAP FI FULL

In [ ]:
#carico shapley value full
path =('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_explanations_full.p')
explanations_shap = dill.load(open(path, 'rb'))

In [ ]:
explanations_shap # non è la stessa struttura dell'altro shap classico

In [ ]:
inst = X_train.iloc[1].values
print('Instance ',inst)
print('True class ',Y_train.iloc[128])
print('Predicted class ',bb.predict(inst.reshape(1, -1)))

### Create the df

In [ ]:
feature_names = X_test.columns
real_feature_names = X_test.columns

In [ ]:
feature_names

In [ ]:
temp={}
i=0
df_viz = pd.DataFrame(columns = ['type', 'name', 'rname', 'min', 'max', 'q1', 'median', 'q3', 'mean',
       'std', 'feature_importance', 'category', 'count', 'inst', 'op', 'thr',
       'is_continuous', 'thr2'])
i_f=0
bool_f= False
for f in feature_names:
    temp[f]= dict()
    temp[f]['feature_importance'] = explanations_shap[i][i_f]
    for lore_p in objs[i].rule.premises:
        if lore_p.att == f:
            op = lore_p.op
            thr = lore_p.thr
            is_continuous = lore_p.is_continuous
            temp[f]['op']= op
            temp[f]['thr']= thr
            temp[f]['is_continuous']= is_continuous
            bool_f= True
    if len(np.unique(X_train[f]))==2:
        type_f = 'categorical'
        temp[f]['type']= type_f
        count=np.unique(X_train[f], return_counts= True)
        temp[f]['count']= count
    else:
        type_f = 'numeric'
        temp[f]['type']= type_f
        min_f = X_train[f].min()
        temp[f]['min']= min_f
        max_f = X_train[f].max()
        temp[f]['max']= max_f
        q_1 =X_train[f].quantile(0.25)
        temp[f]['q1']= q_1
        median =X_train[f].quantile(0.50)
        temp[f]['median']= median
        q_3=X_train[f].quantile(0.75)
        temp[f]['q3']= q_3
        temp[f]['mean'] = X_train[f].mean()
        temp[f]['std']=X_train[f].std()
        if bool_f == True:
            if op == '>=':
                temp[f]['thr_2'] = max_f
            else:
                temp[f]['thr_2'] = min_f
        bool_f=False
    i_f+=1

In [ ]:
temp['PCRIV_FT_20_DLT_PERC_ANNUO_UTILZZ_MEDIO']

In [ ]:
temp

In [ ]:
df_viz = df_viz.from_dict(temp)

In [ ]:
df_viz = df_viz.T.reset_index()

In [ ]:
df_viz=df_viz.rename(columns={'index':'name'})
df_viz['rname']=df_viz['name']

In [ ]:
df_viz

In [ ]:
df_viz['count'][4:5]

In [ ]:
df_viz_exp= df_viz.explode(['count'])
df_viz_exp

In [ ]:
df_viz_exp_2= df_viz_exp.explode(['count'])
df_viz_exp_2

In [ ]:
df_viz_exp_2 = df_viz_exp_2[df_viz_exp_2.count!= 0]

In [ ]:
if count =! Nan:
#copio la riga in un dizionario, correggo i valori di count e category, butto via le vecchie categoriche e riappendo quelle nuove

In [ ]:
(df_viz.set_index(['type', 'name', 'rname', 'min', 'max', 'q1', 'median', 'q3', 'mean',
       'std', 'feature_importance', 'op', 'thr',
       'is_continuous'])
   .apply(lambda x: x.str.split(',[').explode())
   .reset_index())
df_viz